# SQL d'analyse exécuté depuis Python (SQLite en mémoire)

Ce notebook accompagne le module [09 - Extraction de données pour l'analyse](README.md), en particulier [02 - SQL avancé pour l'analyse](02-sql-avance-analyse.md).

**Objectif :** charger un vrai CSV de ventes retail dans une base **SQLite en mémoire** (via `pandas.to_sql`), puis pratiquer les gestes clés du data analyst directement en SQL depuis Python : agrégations, jointures, CTE (`WITH`) et fonctions de fenêtrage (`ROW_NUMBER`, `LAG`).

## 1. Imports et chargement du dataset

On lit le CSV avec pandas, puis on le déverse dans une base SQLite **en mémoire** (`:memory:`) avec `df.to_sql(...)`. La table `ventes` devient alors requêtable comme n'importe quelle base SQL. La fonction utilitaire `q()` exécute une requête et renvoie le résultat en **DataFrame** (`pd.read_sql_query`).

> 💡 Chaque requête suit une **demande métier** : on ne fait pas du SQL pour du SQL, on répond à une question.

In [ ]:
import pandas as pd
import sqlite3

# Chargement du VRAI dataset (chemin relatif depuis ce notebook)
df = pd.read_csv("../../../99-Brief/Data-Analyst/data/ventes_magasins.csv")

# Base SQLite en mémoire + insertion de la table 'ventes'
con = sqlite3.connect(":memory:")
df.to_sql("ventes", con, if_exists="replace", index=False)

def q(sql):
    """Execute une requete SQL sur la base en memoire et renvoie un DataFrame."""
    return pd.read_sql_query(sql, con)

print(f"{len(df):,} lignes chargees dans la table SQLite 'ventes'")
q("SELECT * FROM ventes LIMIT 3")

## 2. Extraction ciblée et indicateurs : WHERE, ORDER BY, GROUP BY, HAVING

> **Demande métier :** « Les 5 plus grosses ventes en magasin à Lille ; puis le CA / panier moyen par catégorie ; puis les villes à plus de 400 000 € de CA. »

Les briques de base : `WHERE` (filtre), `ORDER BY` (tri), `LIMIT` (Top N), puis `SUM`/`COUNT`/`AVG` + `GROUP BY` pour les indicateurs. `WHERE` filtre les lignes **avant** le regroupement ; `HAVING` filtre les groupes **après** agrégation (on ne peut pas mettre `SUM(...)` dans un `WHERE`).

In [ ]:
# Top 5 des plus grosses ventes en magasin a Lille (WHERE + ORDER BY + LIMIT)
display(q("""
SELECT date, ville, categorie, produit, montant
FROM ventes
WHERE ville = 'Lille' AND type = 'Magasin'
ORDER BY montant DESC
LIMIT 5
"""))

# Indicateurs par categorie (GROUP BY)
display(q("""
SELECT
    categorie,
    COUNT(*)               AS nb_ventes,
    ROUND(SUM(montant), 2) AS ca,
    ROUND(AVG(montant), 2) AS panier_moyen
FROM ventes
GROUP BY categorie
ORDER BY ca DESC
"""))

# Filtre APRES agregation (HAVING)
q("""
SELECT ville, ROUND(SUM(montant), 2) AS ca
FROM ventes
GROUP BY ville
HAVING SUM(montant) > 400000
ORDER BY ca DESC
""")

## 3. 🎯 À toi de jouer #1 — un tableau croisé avec CASE WHEN

L'agrégation conditionnelle `SUM(CASE WHEN ... THEN montant ELSE 0 END)` est le **pivot en SQL** : une colonne par valeur. Complète la requête pour obtenir, **par ville**, le CA réalisé en `Magasin` et le CA réalisé en `E-commerce`.

> Rappel : `SUM(CASE WHEN type = 'Magasin' THEN montant ELSE 0 END) AS ca_magasin`

In [ ]:
# TODO : complete la requete pour croiser ville x type de vente.
# Attendu : colonnes ville, ca_magasin, ca_ecommerce (arrondies), triees par ville.

# q("""
# SELECT
#     ville,
#     ROUND(SUM(CASE WHEN type = 'Magasin'    THEN montant ELSE 0 END), 2) AS ca_magasin,
#     ROUND(SUM(CASE WHEN ______ = '________' THEN _______ ELSE 0 END), 2) AS ca_ecommerce
# FROM ventes
# GROUP BY ______
# ORDER BY ville
# """)

## 4. Une jointure : comparer chaque ville à la moyenne nationale

Notre dataset est une table unique. Pour illustrer une **jointure**, on matérialise d'abord une table `ca_ville` (CA par ville) via `to_sql`, puis on la **joint** à une CTE qui calcule la moyenne globale. Résultat : chaque ville avec son écart à la moyenne.

> Un `CROSS JOIN` sur une table à une seule ligne (la moyenne) est la façon propre d'accoler une valeur globale à chaque ligne.

In [ ]:
# On materialise le CA par ville dans une nouvelle table SQL (demo de to_sql + jointure)
ca_ville = (
    df.groupby("ville", as_index=False)["montant"].sum()
      .rename(columns={"montant": "ca"})
)
ca_ville.to_sql("ca_ville", con, if_exists="replace", index=False)

q("""
WITH moyenne AS (
    SELECT AVG(ca) AS ca_moyen FROM ca_ville
)
SELECT
    v.ville,
    ROUND(v.ca, 2)              AS ca,
    ROUND(m.ca_moyen, 2)        AS ca_moyen_national,
    ROUND(v.ca - m.ca_moyen, 2) AS ecart_moyenne
FROM ca_ville v
CROSS JOIN moyenne m
ORDER BY ecart_moyenne DESC
""")

## 5. CTE (`WITH`) : organiser une requête en étapes

> **Demande métier :** « Quelles catégories font un CA supérieur à la moyenne des catégories ? »

Une **CTE** est une sous-requête **nommée**, déclarée avant le `SELECT` principal. On lit la requête de haut en bas comme une suite d'étapes — bien plus lisible que des sous-requêtes imbriquées, et réutilisable plusieurs fois.

In [ ]:
q("""
WITH ca_par_categorie AS (
    SELECT categorie, SUM(montant) AS ca
    FROM ventes
    GROUP BY categorie
)
SELECT categorie, ROUND(ca, 2) AS ca
FROM ca_par_categorie
WHERE ca > (SELECT AVG(ca) FROM ca_par_categorie)
ORDER BY ca DESC
""")

## 6. Window function `ROW_NUMBER` : Top N par groupe

> **Demande métier :** « Donne-moi le top 3 des produits par CA, à l'intérieur de chaque catégorie. »

Impossible avec un simple `GROUP BY` : il faut classer **à l'intérieur** de chaque groupe. `ROW_NUMBER() OVER (PARTITION BY categorie ORDER BY ca DESC)` numérote sans égalité. On filtre `rang <= 3` dans une **CTE** (la window function est calculée après le `WHERE`).

In [ ]:
q("""
WITH ca_produit AS (
    SELECT categorie, produit, SUM(montant) AS ca
    FROM ventes
    GROUP BY categorie, produit
),
classement AS (
    SELECT
        categorie, produit, ca,
        ROW_NUMBER() OVER (PARTITION BY categorie ORDER BY ca DESC) AS rang
    FROM ca_produit
)
SELECT categorie, produit, ROUND(ca, 2) AS ca, rang
FROM classement
WHERE rang <= 3
ORDER BY categorie, rang
""")

## 8. Window function `LAG` : évolution mois par mois

> **Demande métier :** « Quelle est l'évolution du CA mensuel par rapport au mois précédent ? »

`strftime('%Y-%m', date)` extrait le mois. `LAG(ca) OVER (ORDER BY mois)` regarde la ligne précédente ; la première ligne n'a pas de précédent → `NULL`.

> ⚠️ **Piège :** `strftime` n'accepte que des dates ISO `YYYY-MM-DD`. Sur un autre format, SQLite renvoie `NULL` **en silence** (pas d'erreur). Réflexe de validation : on compte d'abord les dates non parsables — ici, ce doit être `0`.

In [ ]:
# Garde-fou : combien de dates SQLite n'arrive-t-il PAS a parser ? (doit valoir 0)
nb_null = q("SELECT COUNT(*) AS n FROM ventes WHERE strftime('%Y-%m', date) IS NULL")["n"].iloc[0]
print("Dates non parsables :", nb_null)

q("""
WITH ca_mensuel AS (
    SELECT strftime('%Y-%m', date) AS mois, SUM(montant) AS ca
    FROM ventes
    GROUP BY mois
)
SELECT
    mois,
    ROUND(ca, 2)                                AS ca,
    ROUND(LAG(ca) OVER (ORDER BY mois), 2)      AS ca_mois_precedent,
    ROUND(ca - LAG(ca) OVER (ORDER BY mois), 2) AS evolution
FROM ca_mensuel
ORDER BY mois
LIMIT 12
""")

## 9. 🎯 À toi de jouer #2 — évolution mensuelle PAR ville avec LAG

Reprends la logique du LAG, mais cette fois **par ville**. Le `PARTITION BY ville` est essentiel : sans lui, `LAG` comparerait le janvier d'une ville au décembre d'une autre.

> Attendu : colonnes `ville`, `mois`, `ca`, `ca_precedent`, triées par `ville` puis `mois`.

In [ ]:
# TODO : complete la CTE et la window function.
# Indice : GROUP BY ville, mois  puis  LAG(ca) OVER (PARTITION BY ville ORDER BY mois)

# q("""
# WITH ca_mensuel AS (
#     SELECT
#         ville,
#         strftime('%Y-%m', date) AS mois,
#         SUM(montant)            AS ca
#     FROM ventes
#     GROUP BY ______, ______
# )
# SELECT
#     ville,
#     mois,
#     ROUND(ca, 2) AS ca,
#     ROUND(LAG(ca) OVER (PARTITION BY ______ ORDER BY ______), 2) AS ca_precedent
# FROM ca_mensuel
# ORDER BY ville, mois
# """)

## 10. 🎯 À toi de jouer #3 — valider un total (contrôle de cohérence)

Un data analyst **vérifie toujours ses totaux**. La somme des `montant` calculée en SQL doit être **égale** (à l'arrondi près) à celle calculée par pandas sur le DataFrame d'origine.

> Complète les deux calculs, puis compare-les.

In [ ]:
# TODO : calcule le CA total des deux facons et verifie qu'elles concordent.

# ca_sql = q("SELECT SUM(montant) AS ca FROM ______")["ca"].iloc[0]
# ca_pandas = df["________"].sum()
# print(f"CA SQL    : {ca_sql:,.2f}")
# print(f"CA pandas : {ca_pandas:,.2f}")
# print("Coherent :", round(ca_sql, 2) == round(ca_pandas, 2))

## 11. Synthèse

Dans ce notebook, on a exécuté du **SQL d'analyse depuis Python** sans installer de serveur : `pandas.to_sql` charge un CSV dans une base **SQLite en mémoire**, et `pd.read_sql_query` renvoie les résultats en DataFrame.

Les gestes clés du module, mis en pratique :

| Je veux… | J'utilise… |
|---|---|
| Filtrer / trier / Top N | `WHERE`, `ORDER BY`, `LIMIT` |
| Produire des indicateurs | `COUNT`, `SUM`, `AVG` + `GROUP BY` |
| Filtrer après agrégation | `HAVING` |
| Tableau croisé | `SUM(CASE WHEN ... THEN ... ELSE 0 END)` |
| Comparer à une valeur globale | jointure / CTE |
| Organiser en étapes | `WITH etape AS (...)` |
| Top N **par groupe** | `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` |
| Évolution vs période précédente | `LAG(ca) OVER (ORDER BY ...)` |

> 🎯 **À graver :** *GROUP BY résume, la window function annote. LAG regarde derrière. Et on valide toujours ses totaux avant de livrer.*

**Pour aller plus loin :** [03 - Extraction multi-sources](03-extraction-multi-sources.md) (API REST + combinaison SQL + Excel avec pandas).